In [4]:
import glob
import os
import datetime
from astropy.table import Table
filtername = 'F480M'
basepath = '/orange/adamginsburg/jwst/w51/'



In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
from jwst.datamodels import dqflags

nircam_short_filters = ['F140M', 'F162M', 'F182M', 'F187N', 'F210M' ]
nircam_long_filters = [ 'F335M',  'F360M','F405N', 'F410M',  'F480M']
miri_filters = ['F560W', 'F770W', 'F1000W',  'F1280W',  'F2100W', ]
def tblname_to_imgname(tblname,  proposal_id='6151'):
    filtername = tblname.split('/')[-1].split('_')[0]
    if filtername.upper() in miri_filters:
        target= 'w51_miri'
    else:        
        target = 'w51'

    nvisits = {'2221': {'brick': 1, 'cloudc': 2},
                '1182': {'brick': 2},
                '6151': {'w51': 1, 'w51_miri': 2}
                }
    field_to_reg_mapping = {'2221': {'001': 'brick', '002': 'cloudc'},
                            '1182': {'004': 'brick'},
                            '6151': {'001': 'w51', '002':'w51_miri'}}[proposal_id]
    reg_to_field_mapping = {v:k for k,v in field_to_reg_mapping.items()}
    field = reg_to_field_mapping[target]


    
    module = tblname.split('/')[-1].split('_')[1]
    visit = tblname.split('/')[-1].split('_')[2]
    visitid = visit[5:]
    vgroup = tblname.split('/')[-1].split('_')[3]
    vgroupid = vgroup[6:]
    exposure = tblname.split('/')[-1].split('_')[4]
    expid = exposure[3:]
    imgname = f'{basepath}/{filtername.upper()}/pipeline/jw0{proposal_id}{field}{visitid}_{vgroupid}_{expid}_{module}_cal.fits'


    return imgname
def load_data(filename):
    fh = fits.open(filename)
    im1 = fh
    data = im1['SCI'].data
    try:
        wht = im1['WHT'].data
    except KeyError:
        wht = None
    err = im1['ERR'].data
    instrument = im1[0].header['INSTRUME']
    telescope = im1[0].header['TELESCOP']
    obsdate = im1[0].header['DATE-OBS']
    return fh, im1, data, wht, err, instrument, telescope, obsdate
def filter_by_nmatch(tbls_merged, tbls, tolerance=0):

    for jj, tbl_merged in enumerate(tbls_merged):
        # convert skycoord of tbl_merged to pixel coordinates in each tbl in tbls
        skycoord_merged = tbl_merged['skycoord']
        for ii, tbl in enumerate(tbls):
            img_name = tblname_to_imgname(tblfns[ii])
            fh, im1, img_data, wht, err, instrument, telescope, obsdate = load_data(img_name)
            wcs = WCS(im1[1].header)
            pixcoord_merged = wcs.world_to_pixel(skycoord_merged)
                    ㅇㄹㅇㄹㅇㄹㄹㅇㄹㅇㄹㅇㄹㅇㄹㅇㄹㅇㅌㄹㅇㄹㅇsfdsfdsfadf      fdfdfdfdfddfdfdfdfdfdffdfdfdfdfdfdfdfdffdf`dfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdfdq = im1['DQ'].data

            
            # check whether pixel coordiantes of tbl_merged fall within the image field of view
         
            is_saturated = dq & dqflags.pixel['SATURATED']
            is_hot = dq & dqflags.pixel['HOT']
            is_dead = dq & dqflags.pixel['DEAD']
            is_really_saturated = is_saturated & ~is_hot & ~is_dead
            is_nan_but_is_really_saturated = np.isnan(img_data) & is_really_saturated

            img_shape = img_data.shape
            finite_coord = np.isfinite(pixcoord_merged[0]) & np.isfinite(pixcoord_merged[1])
            xpix = pixcoord_merged[0].astype(int)
            ypix = pixcoord_merged[1].astype(int)
            in_fov = finite_coord & (pixcoord_merged[0] >= 0) & (pixcoord_merged[0] < img_shape[1]) & (pixcoord_merged[1] >= 0) & (pixcoord_merged[1] < img_shape[0])
            valid_pixel = np.zeros(in_fov.shape, dtype=bool)
            valid_pixel[in_fov] = np.isfinite(img_data[ypix[in_fov], xpix[in_fov]]) | is_really_saturated[ypix[in_fov], xpix[in_fov]]
            in_fov = in_fov & valid_pixel
            in_fov_int = in_fov.astype(int)
            if ii == 0:
                in_fov_all = in_fov_int
            else:
                in_fov_all = in_fov_all + in_fov_int

        nmatch_max = in_fov_all
        nmatch = tbl_merged['nmatch']
        from_sat_cat = tbl_merged['from_sat_catalog']
        from_sat_cat = from_sat_cat.astype(bool)
        keep = (nmatch >= nmatch_max - tolerance) | from_sat_cat
        tbl_merged_cut = tbl_merged[keep]
        print(f"Number of sources in merged table: {len(tbl_merged)}")
        print(f"Number of sources in merged table with nmatch >= {nmatch_max}: {len(tbl_merged_cut)}")  
        print(f"Number of sources in merged table with nmatch >= {nmatch_max - tolerance}: {len(tbl_merged[nmatch >= nmatch_max - tolerance])}")    
        #save the cut table to a new file
        if tolerance == 0:
            label = 'grade_a'
        elif tolerance ==1:
            label = 'grade_b'
        tbl_merged_cut.write(tblfns_merged[jj].replace('.fits', f'_nmatch_cut_{label}.fits'), overwrite=True)
    
            


In [6]:
filternames=  miri_filters
for filtername in filternames:
    tblfns = glob.glob(f'{basepath}/{filtername.upper()}/*daophot_combined_with_satstars.fits')
    for tbl in tblfns:
        print(tbl)
        print('last modified:', datetime.datetime.fromtimestamp(os.path.getmtime(tbl)))

    tblfns_merged = glob.glob(f'/orange/adamginsburg/jwst/w51/catalogs/{filtername.lower()}_*_indivexp_merged_dao_after_merger_combined_with_satstars.fits')
    print('last modified:', datetime.datetime.fromtimestamp(os.path.getmtime(tblfns_merged[0])))

    tbls = [Table.read(tblfn) for tblfn in tblfns]
    tbls_merged = [Table.read(tblfn) for tblfn in tblfns_merged]


    filter_by_nmatch(tbls_merged, tbls, tolerance=0)
    filter_by_nmatch(tbls_merged, tbls, tolerance=1)

/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit002_vgroup0210b_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:28
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup02101_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:26
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup02101_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:25
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup0210b_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:25
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit002_vgroup02101_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:28
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup02101_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:24
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit002_vgroup02101_exp00001_

Set DATE-AVG to '2024-09-08T12:32:25.589' from MJD-AVG.
Set DATE-END to '2024-09-08T12:32:32.527' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.702250 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291148460.660 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T10:15:23.005' from MJD-AVG.
Set DATE-END to '2024-09-08T10:15:29.943' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.806250 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291912377.206 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/astropy/wcs/wcsapi/fitswcs.py:367: UserWarning: 'WCS.all_world2pix' failed to converge to the requested accuracy.
After 20 iterations, the solution is diverging at least for one input point.
  warnings.warn(str(e))
Set DATE-AVG to '2024-09-08T10:13:45.853' from MJD-AVG.
Set DATE-END to '2024-09-08T10:13:52.791' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.807478 from OBSGEO-[XYZ].
Set OBSGEO-H to 129192142

Number of sources in merged table: 37218
Number of sources in merged table with nmatch >= [5 5 6 ... 6 6 6]: 4888
Number of sources in merged table with nmatch >= [5 5 6 ... 6 6 6]: 4864


Number of sources in merged table: 37218
Number of sources in merged table with nmatch >= [5 5 6 ... 6 6 6]: 9001
Number of sources in merged table with nmatch >= [4 4 5 ... 5 5 5]: 8978
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit002_vgroup02103_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-31 14:26:42
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit002_vgroup0210d_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-31 14:26:42
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit001_vgroup0210d_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-31 14:26:39
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit002_vgroup0210d_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-31 14:26:42
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit002_vgroup02103_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-31 14:26:42
/orange/adamginsburg/jwst/w51//F770W/f770w_mirim

Set DATE-AVG to '2024-09-08T11:54:02.295' from MJD-AVG.
Set DATE-END to '2024-09-08T11:54:09.233' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.731399 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291362083.380 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T12:40:31.285' from MJD-AVG.
Set DATE-END to '2024-09-08T12:40:38.223' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.696102 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291103450.217 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T11:07:49.947' from MJD-AVG.
Set DATE-END to '2024-09-08T11:07:56.884' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.766467 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291619585.186 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T12:42:11.190' from MJD-AVG.
Set DATE-END to '2024-09-08T12:42:18.127' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.694837 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291094193.500 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE

...

Number of sources in merged table: 19591
Number of sources in merged table with nmatch >= [6 6 6 ... 6 6 6]: 1833
Number of sources in merged table with nmatch >= [6 6 6 ... 6 6 6]: 1805
Number of sources in merged table: 19591
Number of sources in merged table with nmatch >= [6 6 6 ... 6 6 6]: 3908
Number of sources in merged table with nmatch >= [5 5 5 ... 5 5 5]: 3883
/orange/adamginsburg/jwst/w51//F1280W/f1280w_mirimage_visit001_vgroup0210h_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:17
/orange/adamginsburg/jwst/w51//F1280W/f1280w_mirimage_visit002_vgroup0210h_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:21
/orange/adamginsburg/jwst/w51//F1280W/f1280w_mirimage_visit001_vgroup02107_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:18
/orange/adamginsburg/jwst/w51//F1280W/f1280w_mirimage_visit001_vgroup0210h_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:18
/orang

Set DATE-AVG to '2024-09-08T11:22:51.833' from MJD-AVG.
Set DATE-END to '2024-09-08T11:22:58.771' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.755061 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291535770.906 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T12:55:44.245' from MJD-AVG.
Set DATE-END to '2024-09-08T12:55:51.182' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.684544 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291018878.595 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T10:39:59.356' from MJD-AVG.
Set DATE-END to '2024-09-08T10:40:06.294' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.787589 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291774951.446 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T11:24:28.986' from MJD-AVG.
Set DATE-END to '2024-09-08T11:24:35.923' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.753832 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291526744.918 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE

Number of sources in merged table: 13889
Number of sources in merged table with nmatch >= [6 6 6 ... 6 7 4]: 1439
Number of sources in merged table with nmatch >= [6 6 6 ... 6 7 4]: 1407
Number of sources in merged table: 13889
Number of sources in merged table with nmatch >= [6 6 6 ... 6 7 4]: 3243
Number of sources in merged table with nmatch >= [5 5 5 ... 5 6 3]: 3213
/orange/adamginsburg/jwst/w51//F2100W/f2100w_mirimage_visit001_vgroup0210j_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:23
/orange/adamginsburg/jwst/w51//F2100W/f2100w_mirimage_visit001_vgroup0210j_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:23
/orange/adamginsburg/jwst/w51//F2100W/f2100w_mirimage_visit001_vgroup02109_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:23
/orange/adamginsburg/jwst/w51//F2100W/f2100w_mirimage_visit001_vgroup02109_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-31 12:58:23
/orang

Set DATE-AVG to '2024-09-08T11:27:46.041' from MJD-AVG.
Set DATE-END to '2024-09-08T11:27:52.979' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.751340 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291508438.907 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T11:30:57.530' from MJD-AVG.
Set DATE-END to '2024-09-08T11:31:04.467' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.748918 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291490652.156 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T10:43:16.349' from MJD-AVG.
Set DATE-END to '2024-09-08T10:43:23.286' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.785099 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291756623.231 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T10:44:50.748' from MJD-AVG.
Set DATE-END to '2024-09-08T10:44:57.685' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.783905 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291747841.087 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE

Number of sources in merged table: 1056
Number of sources in merged table with nmatch >= [6 6 5 ... 7 6 4]: 185
Number of sources in merged table with nmatch >= [6 6 5 ... 7 6 4]: 150
Number of sources in merged table: 1056
Number of sources in merged table with nmatch >= [6 6 5 ... 7 6 4]: 342
Number of sources in merged table with nmatch >= [5 5 4 ... 6 5 3]: 308
